[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcosma/RULER/blob/main/tutorial/reproduce_breast_cancer.ipynb)

# Reproducing the paper's Breast Cancer results

This notebook reproduces the paper's result for **Breast Cancer**, the
smallest of its tabular datasets: the pre-unlearning `m4` averaged over all ten
training seeds, and `m1`-`m4` for all four unlearning methods, in the shape of
the paper's Table 1.

It uses the pre-trained models in `checkpoints/` and computes every metric with
the `ruler` library. It runs in about ten seconds on a CPU.

> **Running this.** Click the *Open in Colab* badge above and choose
> *Runtime -> Run all*, or clone the repository and run it locally:
> `git clone https://github.com/gcosma/RULER.git`

In [1]:
# Locate the RULER repository, cloning it if we are not already inside a clone
# (so this works in Google Colab as well as from a local checkout).
import subprocess, sys
from pathlib import Path

def _is_repo(p):
    return (p / "ruler").is_dir() and (p / "checkpoints").is_dir()

root = Path.cwd()
while root != root.parent and not _is_repo(root):
    root = root.parent
if not _is_repo(root):
    target = Path.cwd() / "RULER"
    if not _is_repo(target):
        print("Cloning gcosma/RULER ...")
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/gcosma/RULER.git", str(target)], check=True)
    root = target
assert _is_repo(root), f"could not find or fetch the RULER repository (looked at {root})"
sys.path.insert(0, str(root))
CHECKPOINTS = root / "checkpoints"
print("using RULER repository at:", root)

using RULER repository at: /home/user/RULER


## Data, models and partition — exactly as the paper

Breast Cancer from scikit-learn (split seed 999, scaler on train); the
`d -> 128 -> 128 -> 2` architecture the checkpoints use, read at the penultimate
layer; and the forget/retain split at seed 999, forget fraction 5%.

In [2]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

bundle = load_breast_cancer()
x = bundle.data.astype(np.float32)
y = bundle.target.astype(np.int64)
x_train, _, y_train, _ = train_test_split(x, y, test_size=0.2, random_state=999, stratify=y)
x_train = StandardScaler().fit(x_train).transform(x_train).astype(np.float32)
n = len(x_train)

class TabularMLP(nn.Module):
    def __init__(self, d, h=128, o=2, p=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, h), nn.ReLU(), nn.Dropout(p),
            nn.Linear(h, h), nn.ReLU(), nn.Dropout(p),
            nn.Linear(h, o),
        )
    def forward(self, z):
        return self.net(z)

def load_checkpoint(name, input_dim=30):
    state = torch.load(CHECKPOINTS / name, map_location="cpu", weights_only=True)
    model = TabularMLP(input_dim)
    model.net.load_state_dict({k[len("net."):]: v for k, v in state.items()})
    return model.eval()

@torch.no_grad()
def embed(model, xs):
    model.eval()
    return model.net[:5](torch.as_tensor(xs)).numpy()   # penultimate layer

size = max(10, int(0.05 * n))
rng = np.random.RandomState(999)
forget_idx = np.sort(rng.choice(n, size, replace=False))
retain_idx = np.setdiff1d(np.arange(n), forget_idx, assume_unique=True)
print(f"{n} training records; {len(forget_idx)} to forget, {len(retain_idx)} to retain")

455 training records; 22 to forget, 433 to retain


## The four unlearning methods

The paper's four approximate methods, with its constants (learning rate 5e-4;
5 epochs for Gradient Ascent, 10 for the rest; alpha = 0.6, temperature = 2.0;
seed 100). These produce the *unlearned* models that `m1`-`m3` compare against
the oracle. RULER itself is method-agnostic — this is the user's own experiment
code, kept here so the reproduction is self-contained.

In [3]:
import copy
import torch.nn.functional as F
from torch import optim

LR, EPOCHS, GA_EPOCHS, ALPHA, TEMP, SEED = 5e-4, 10, 5, 0.6, 2.0, 100

def _kl(student_logits, teacher_logits, t):
    return F.kl_div(F.log_softmax(student_logits / t, 1),
                    F.softmax(teacher_logits / t, 1), reduction="batchmean") * t * t

def unlearn(method, original, xf, yf, xr, yr):
    torch.manual_seed(SEED)
    student = copy.deepcopy(original).train()
    opt = optim.Adam(student.parameters(), lr=LR)
    ce = nn.CrossEntropyLoss()
    xf, yf, xr, yr = map(torch.as_tensor, (xf, yf, xr, yr))
    if method == "Gradient Ascent":
        for _ in range(GA_EPOCHS):
            opt.zero_grad(); (-ce(student(xf), yf)).backward(); opt.step()
    elif method == "NegGrad+":
        for _ in range(EPOCHS):
            opt.zero_grad()
            (ALPHA * ce(student(xr), yr) - (1 - ALPHA) * ce(student(xf), yf)).backward(); opt.step()
    elif method == "Fine-Tuning":
        for _ in range(EPOCHS):
            opt.zero_grad(); ce(student(xr), yr).backward(); opt.step()
    elif method == "SCRUB":
        teacher = copy.deepcopy(original).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        with torch.no_grad():
            tf = teacher(xf)
        for _ in range(EPOCHS):
            opt.zero_grad()
            with torch.no_grad():
                tr = teacher(xr)
            (ALPHA * _kl(student(xr), tr, TEMP) - (1 - ALPHA) * _kl(student(xf), tf, TEMP)).backward()
            opt.step()
    return student.eval()

print("four unlearning methods defined")

four unlearning methods defined


## Run the grid: 10 seeds x 4 methods, metrics from the library

For each seed we load the original and oracle checkpoints, run each method to
get the unlearned model, and let `all_metrics` compute `m1`-`m4`. The M2
baseline uses at most 500 retain records (paper Section 4.3, sampled at seed
42); Breast Cancer's retain set is smaller, so all of it is used.

In [4]:
from ruler import all_metrics, m4

METHODS = ["Gradient Ascent", "NegGrad+", "Fine-Tuning", "SCRUB"]
M2_SUBSAMPLE, SUBSAMPLE_SEED = 500, 42

pre_unlearning = []
agg = {m: {"m1": [], "m2": [], "m3": [], "m4": []} for m in METHODS}

for seed in range(10):
    original = load_checkpoint(f"breast_cancer_seed{seed}_orig.pt")
    oracle   = load_checkpoint(f"breast_cancer_seed{seed}_oracle_ff05.pt")

    pre_unlearning.append(m4(embed(original, x_train[forget_idx]),
                             embed(original, x_train[retain_idx])))

    o_forget, o_retain = embed(oracle, x_train[forget_idx]), embed(oracle, x_train[retain_idx])
    orig_forget = embed(original, x_train[forget_idx])
    keep = (np.arange(len(retain_idx)) if len(retain_idx) <= M2_SUBSAMPLE
            else np.random.RandomState(SUBSAMPLE_SEED).choice(len(retain_idx), M2_SUBSAMPLE, replace=False))

    for method in METHODS:
        u = unlearn(method, original,
                    x_train[forget_idx], y_train[forget_idx],
                    x_train[retain_idx], y_train[retain_idx])
        scores = all_metrics(embed(u, x_train[forget_idx]), embed(u, x_train[retain_idx])[keep],
                             o_forget, o_retain[keep], orig_forget)
        for k in ("m1", "m2", "m3", "m4"):
            agg[method][k].append(scores[k])

print(f"done: {len(pre_unlearning)} seeds x {len(METHODS)} methods")

done: 10 seeds x 4 methods


## Results

In [5]:
print(f"Pre-unlearning M4 (original model), mean over 10 seeds = "
      f"{np.mean(pre_unlearning):.3f} +/- {np.std(pre_unlearning):.3f}"
      f"   (paper: Breast Cancer around 0.60)\n")

print(f"{'Method':<16}{'M1':>10}{'M2':>12}{'M3':>12}{'M4':>10}")
print("-" * 60)
for method in METHODS:
    a = agg[method]
    print(f"{method:<16}{np.mean(a['m1']):>+10.4f}{np.mean(a['m2']):>+12.5f}"
          f"{np.mean(a['m3']):>+12.5f}{np.mean(a['m4']):>+10.4f}")

# Directional consistency across seeds (descriptive, not an inferential test).
# The paper's significance result is a mixed-effects model across all ten
# datasets; on one dataset the honest summary is how consistently the sign holds.
print()
for method in METHODS:
    n_neg = sum(1 for v in agg[method]["m2"] if v < 0)
    print(f"  {method:<16} M2 < 0 in {n_neg}/10 seeds")

Pre-unlearning M4 (original model), mean over 10 seeds = 0.611 +/- 0.021   (paper: Breast Cancer around 0.60)

Method                  M1          M2          M3        M4
------------------------------------------------------------
Gradient Ascent    +0.9937    -0.00163    -0.00221   +0.6030
NegGrad+           +0.9914    -0.00403    -0.00448   +0.6006
Fine-Tuning        +0.9955    -0.00049    -0.00046   +0.6096
SCRUB              +0.9933    -0.00259    -0.00262   +0.6079

  Gradient Ascent  M2 < 0 in 9/10 seeds
  NegGrad+         M2 < 0 in 10/10 seeds
  Fine-Tuning      M2 < 0 in 8/10 seeds
  SCRUB            M2 < 0 in 10/10 seeds


## Reading the table

- **Pre-unlearning M4 around 0.61** reproduces the paper's Breast Cancer value
  (~0.60): the original model memorises these records.
- **Every method has M2 < 0** — the forget records sit further from the oracle
  than the retain records do. This is the paper's central finding: residual
  memorisation survives methods that pass output-level checks.
- **Every method keeps M4 around 0.60** — the memorisation the metric sees
  before unlearning is still there afterwards.
- **NegGrad+ shows the largest M2 magnitude**, matching the paper's ordering.

These are the values for one dataset; the paper's Table 1 pools all ten
tabular datasets and tests the sign statistically (significant in 10 of 12
conditions). Here, on the smallest dataset, the direction is already
clear — and every number came from the `ruler` library.